# Notebook 02: Real Arm Baselines

**Purpose**: Run all 7 Family A demo-selection conditions on all real datasets, ID and OOD test sets, 2 models, 5 seeds, at both `k_primary` and `k_sensitivity` demo counts.

**Compute budget**: 7 conditions x 5 seeds x 1000 rows x N datasets x 2 models x 2 k-values LLM calls (~12-24h wall time at k_primary alone; the k_sensitivity sweep roughly doubles this, minus zero-shot which is k-independent and only ever run once).

## Why this notebook exists (thesis framing)

This notebook answers **RQ2**: *which kinds of demonstration diversity matter most for OOD performance?* It is also the primary evidence source for **RQ1** (alongside Notebook 01's pilot) and feeds the "best-performing protocol" selection used by Notebook 03's faithfulness evaluation and Notebook 05/06's SATA comparisons.

**Why compare 7 conditions rather than just testing SATA directly?** The lit review's Task 2 (Section 3) lays out four baselines specifically to isolate *which component* of the intervention is doing the work: (1) random demonstrations, to check whether demonstration design matters *at all*; (2) similarity-based retrieval (Liu et al. 2022, "What makes good in-context examples for GPT-3?"), to check whether the current best-practice retrieval method already solves OOD tabular tasks; (3) the four diversity protocols, to check whether principled, hand-designed selection closes the gap without any learning; and (4) SATA (Notebook 06), to check whether *learned, query-conditioned* reweighting adds anything beyond hand-designed diversity. This notebook covers baselines 1–3; SATA is evaluated on top of them in Notebook 06.

**Why also sweep k?** `configs/default.yaml`'s `k_sensitivity` existed as an unused placeholder until this sweep was added — a condition ranking that only holds at one specific demo count (k=8) would be a much weaker RQ2 result than one that's stable across k. All 4 downstream consumers of this notebook's summary (Notebook 03, 05, 06, 08) key off `k == config.k_primary` for their "headline" numbers; the k_sensitivity rows are an additional robustness check, not a replacement.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Conditions (7 total)

1. Zero-shot
2. Random-k
3. Similarity-k (all-MiniLM-L6-v2 cosine similarity)
4. Label diversity (stratified k/2 per class)
5. Feature-range diversity (quantile-bin coverage on top-3 continuous features)
6. Rule diversity (depth-3 decision tree leaf coverage)
7. Counter-spurious diversity (minority-cell over-sampling)

### Why these specific four diversity protocols, and why they're expected to differ

Two empirical findings from the lit review directly motivate this design:

- **Min et al. (2022), "Rethinking the role of demonstrations"** — across twelve LLMs including GPT-3, randomly corrupting demonstration *labels* barely hurts ICL accuracy. Ground-truth input–label mapping isn't the key signal; what matters is the **label space**, the **input-text distribution** the demonstrations sample, and the **input-label format** the demonstrations span. This is why demonstration design has room to operate on *which regions of input space and label space* the demos cover — that's exactly what protocols 4–7 manipulate — rather than needing to retrieve "more correct" examples.
- **Harutyunyan et al. (2024) and Chen et al. (2026)** — in-context learners are themselves susceptible to spurious correlations carried in the demonstration set, and this is a distinct failure mode from the base model's pretraining biases. Protocol 7 (counter-spurious) exists specifically to counteract this at the demonstration level, by denying the model the option of relying on the shortcut.

Each protocol targets a **different point in the covariate/concept shift taxonomy** (Lit-review §2.1.1), which is the basis for RQ2's success criterion:
- **Label diversity** guards against *label (prior-probability) shift* — a skewed class prior in the demo pool would bias the model's implicit prior even when `P(y|x)` hasn't moved.
- **Feature-range diversity** targets *covariate shift* (`P(x)` moves, `P(y|x)` stable) — forcing demos to span the full input range denies the model a narrow, spuriously-correlated cluster to anchor on.
- **Rule diversity** and **counter-spurious diversity** both target *concept shift* (`P(y|x)` itself moves) — the harder regime, per WHYSHIFT (Liu et al. 2023, Lit-review §2.1.3), because no amount of reweighting recovers a moved conditional without information about *how* the decision rule varies across regimes.

**RQ2 succeeds** only if at least one protocol beats random selection *and* different protocols win on different shift types — i.e. the interaction effect is the point, not any single protocol dominating everywhere. A protocol that's uniformly best (or uniformly no better than random) would actually be a weaker result for RQ2 than protocols that trade off differently across shift types.

In [2]:
from src.selection import (
    random_select,
    similarity_select,
    label_diversity,
    feature_range,
    rule_diversity,
    counter_spurious,
)

CONDITIONS = [
    'zero_shot', 'random', 'similarity', 'label_diversity',
    'feature_range', 'rule_diversity', 'counter_spurious',
]

## Prompt template & LLM runner

See `src/inference/prompts.py::build_classification_prompt` and `src/inference/llm_runner.py::VLLMRunner`.

SamplingParams: `logprobs=True, max_tokens=1, temperature=0`. Top token not in the label set -> log as `INVALID`, exclude from accuracy but include in the count.

**Why constrained single-token decoding instead of free-text generation?** This makes the prediction a clean forced choice between exactly the label tokens, so `logprob_0`/`logprob_1` are directly comparable across every condition/dataset/model combination and feed straight into the uncertainty analysis in Notebook 07 (R-AUC, F1@95%) — see `src/inference/llm_runner.py::get_confidence`'s constrained softmax. `temperature=0` makes predictions deterministic given the prompt, so any variation in results across seeds is attributable entirely to which demonstrations were selected, not to sampling noise in generation.

In [3]:
import json

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.data.tableshift_loader import (
    SELECTED_DATASETS, TASK_DESCRIPTIONS, load_codebook, select_top_features,
)
from src.data.serialisation import serialise_row, ordered_feature_names
from src.inference.llm_runner import VLLMWorkerRunner
from src.inference.prompts import build_classification_prompt
from src.selection.rule_diversity import fit_leaf_tree
from src.selection.counter_spurious import find_spurious_proxy_features
from src.utils.results_schema import append_results, new_results_frame, load_results

# TableShift's domain_split_varname per dataset (tableshift/configs/benchmark_configs.py)
# — the variable that defines the OOD shift, used by the counter-spurious protocol
# to find features that proxy for it.
RESULTS_PATH = resolve_path('results/real_arm_baselines.parquet')

# k-sensitivity sweep: run every condition at both k_primary (the headline
# number) and k_sensitivity, on every dataset -- not just the one dataset the
# config comment originally scoped this to. K_VALUES[0] must stay k_primary:
# zero-shot dedup and the "headline" summary filter below both key off it.
K_VALUES = [config.k_primary, config.k_sensitivity]


def prepare_condition_artifacts(dataset_name, train_pool, feature_cols, test_id, test_ood, k, codebook=None):
    """One-time-per-dataset setup shared across seeds/queries/k-values for the
    feature_range, rule_diversity, and counter_spurious protocols.

    `k` here should be max(K_VALUES): similarity ranks are k-independent (a
    cosine-similarity argsort doesn't change when you later take a smaller
    prefix of it), so computing the top-max(K_VALUES) once and slicing per
    k in select_demos avoids re-embedding the same pool/queries once per k.
    """
    artifacts = {}

    # Feature-range: top-3 continuous (numeric, >10 unique values) features by MI.
    continuous_cols = [
        c for c in feature_cols
        if pd.api.types.is_numeric_dtype(train_pool[c]) and train_pool[c].nunique() > 10
    ]
    if continuous_cols:
        artifacts['top3_continuous'] = select_top_features(
            train_pool[continuous_cols + ['label']], n_features=min(3, len(continuous_cols))
        )
    else:
        artifacts['top3_continuous'] = feature_cols[:3]

    # Rule diversity: depth-3 tree fit on the pool.
    artifacts['tree'] = fit_leaf_tree(train_pool, feature_cols)

    # Counter-spurious: proxy feature most correlated with both the label and
    # the actual ID->OOD shift. TableShift's own domain-split covariate (e.g.
    # race/geography/year) is deliberately excluded from X -- it's the exact
    # variable tableshift thresholds to build the ood split, so a model can't
    # just read it directly -- and extract_tableshift_cache.py, which only
    # ever saves X, never had it to cache. Proxy against literal
    # train-vs-OOD-test row membership instead: a feature correlated with
    # *that* carries the same shift signal the raw covariate would have,
    # without needing the excluded column.
    shift_frame = pd.concat(
        [
            train_pool[feature_cols + ['label']].assign(_is_ood=0),
            test_ood[feature_cols + ['label']].assign(_is_ood=1),
        ],
        ignore_index=True,
    )
    proxy_features = find_spurious_proxy_features(shift_frame, feature_cols, 'label', '_is_ood', top_n=3)
    proxy_col = proxy_features[0] if proxy_features else feature_cols[0]
    proxy_high = train_pool[proxy_col] > train_pool[proxy_col].median()
    artifacts['proxy_col'] = proxy_col
    artifacts['proxy_majority_label'] = train_pool.loc[proxy_high, 'label'].mode().iloc[0]

    # Similarity is deterministic (no seed dependency) and reuses the same
    # pool/query embeddings across every one of the 5 seeds -- precompute it
    # once here (one batched encode() call per split) instead of re-embedding
    # one query at a time, identically, on every seed's pass through the data.
    pool_texts = [
        serialise_row(
            {f: train_pool.loc[i, f] for f in ordered_feature_names({f: train_pool.loc[i, f] for f in feature_cols})},
            label=str(int(train_pool.loc[i, 'label'])),
            codebook=codebook,
        )
        for i in train_pool.index
    ]
    similarity_demo_ids = {}
    for environment, test_df in [('id', test_id), ('ood', test_ood)]:
        query_texts = [
            serialise_row(
                {f: row[f] for f in ordered_feature_names({f: row[f] for f in feature_cols})},
                codebook=codebook,
            )
            for _, row in test_df.iterrows()
        ]
        local_idx_per_query = similarity_select.select_batch(pool_texts, query_texts, k)
        similarity_demo_ids[environment] = [
            [train_pool.index[i] for i in local_idx] for local_idx in local_idx_per_query
        ]
    artifacts['similarity_demo_ids'] = similarity_demo_ids

    return artifacts


def select_demos(condition, pool, query, k, seed, feature_cols, artifacts, environment, query_id):
    if condition == 'zero_shot':
        return []
    if condition == 'random':
        return random_select.select(pool, query, k, seed)
    if condition == 'similarity':
        # artifacts['similarity_demo_ids'] holds the top-max(K_VALUES) ranking;
        # a smaller k's top-k is just its prefix (see prepare_condition_artifacts).
        return artifacts['similarity_demo_ids'][environment][query_id][:k]
    if condition == 'label_diversity':
        return label_diversity.select(pool, query, k, seed)
    if condition == 'feature_range':
        return feature_range.select(pool, query, k, seed, top_features=artifacts['top3_continuous'])
    if condition == 'rule_diversity':
        return rule_diversity.select(pool, query, k, seed, feature_cols=feature_cols, tree=artifacts['tree'])
    if condition == 'counter_spurious':
        return counter_spurious.select(
            pool, query, k, seed, proxy_col=artifacts['proxy_col'], proxy_majority_label=artifacts['proxy_majority_label']
        )
    raise ValueError(f"Unknown condition: {condition}")


def build_demo_lines(pool, demo_ids, feature_cols, codebook=None):
    lines = []
    for i in demo_ids:
        row = pool.loc[i]
        ordered = ordered_feature_names({f: row[f] for f in feature_cols})
        lines.append(serialise_row({f: row[f] for f in ordered}, label=str(int(row['label'])), codebook=codebook))
    return lines


def build_query_line(query, feature_cols, codebook=None):
    ordered = ordered_feature_names({f: query[f] for f in feature_cols})
    return serialise_row({f: query[f] for f in ordered}, codebook=codebook)


try:
    import vllm  # noqa: F401
    VLLM_AVAILABLE = True
except ImportError:
    VLLM_AVAILABLE = False
    print("vLLM not installed in this environment — skipping real-arm inference. "
          "Run this notebook on a GPU box with vllm + the model weights available.")

# Model-outer / dataset-inner so each model's weights load exactly once
# (vLLM model load is the expensive step, not iterating datasets/conditions/seeds).
# Compute budget per the docstring above, now roughly doubled by the k=16
# sensitivity sweep (zero-shot is deduped across k, see below) — nested tqdm
# bars below give a glanceable readout of exactly which
# (model, dataset, k, condition, seed) is running.
models_to_run = config.base_llms if VLLM_AVAILABLE else []

# Resume support: append_results always appends and never dedupes, and this
# loop had no way to know what's already been done -- so a restart (e.g.
# after a walltime kill partway through this ~12-24h run) would otherwise
# redo *and duplicate* every combo from scratch. Skip any
# (model, dataset, condition, seed, k) already present in RESULTS_PATH instead,
# so re-running (interactively, or via a resubmitted gadi_run_notebook.pbs
# batch job) picks up exactly where the last run left off. zero_shot always
# has 0 demos regardless of which k in K_VALUES is being swept, so its
# combo-key uses k=0 -- this also means the k=16 pass naturally skips
# re-running zero_shot once the k=8 pass has already produced that row.
if RESULTS_PATH.exists():
    _existing_results = load_results(RESULTS_PATH)
    completed_combos = set(
        zip(_existing_results['model'], _existing_results['dataset'],
            _existing_results['method'], _existing_results['seed'].astype(int),
            _existing_results['k'].astype(int))
    )
else:
    completed_combos = set()

model_bar = tqdm(models_to_run, desc="Models", position=0)
for model_cfg in model_bar:
    model_bar.set_postfix(model=model_cfg.name)
    # VLLMWorkerRunner (not VLLMRunner): runs vLLM in a separate OS subprocess
    # so its GPU memory is guaranteed to be released on shutdown() -- vLLM's
    # in-process engine does not reliably free GPU memory after explicit
    # teardown (confirmed: a second in-process model still failed to init
    # with "Free memory ... is less than desired GPU memory utilization"),
    # which matters here since this loop constructs one runner per model.
    runner = VLLMWorkerRunner(model_cfg.path, **vars(config.vllm))

    dataset_bar = tqdm(SELECTED_DATASETS, desc="Datasets", position=1, leave=False)
    for dataset_name in dataset_bar:
        dataset_bar.set_postfix(dataset=dataset_name)
        data_dir = resolve_path(config.paths.data_real) / dataset_name
        train_pool = pd.read_parquet(data_dir / 'train_pool.parquet')
        test_id = pd.read_parquet(data_dir / 'test_id.parquet')
        test_ood = pd.read_parquet(data_dir / 'test_ood.parquet')
        feature_cols = json.load(open(data_dir / 'feature_list.json'))
        label_tokens = tuple(json.load(open(data_dir / 'label_tokens.json')))
        codebook = load_codebook(data_dir)
        task_description, _task_noun, label_meaning_0, label_meaning_1 = TASK_DESCRIPTIONS[dataset_name]

        artifacts = prepare_condition_artifacts(
            dataset_name, train_pool, feature_cols, test_id, test_ood, max(K_VALUES), codebook=codebook
        )

        k_bar = tqdm(K_VALUES, desc="k", position=2, leave=False)
        for k in k_bar:
            k_bar.set_postfix(k=k)
            condition_bar = tqdm(CONDITIONS, desc="Conditions", position=3, leave=False)
            for condition in condition_bar:
                # Zero-shot has no demos, so it's identical at every k in
                # K_VALUES -- only run it at the primary k, not once per
                # sweep value (the completed_combos key above also catches
                # this on resume, but skip explicitly here too rather than
                # relying solely on that to avoid the wasted inference calls).
                if condition == 'zero_shot' and k != K_VALUES[0]:
                    continue
                condition_bar.set_postfix(condition=condition)
                seed_bar = tqdm(config.seed_accuracy, desc="Seeds", position=4, leave=False)
                for seed in seed_bar:
                    seed_bar.set_postfix(seed=int(seed))
                    combo_k = 0 if condition == 'zero_shot' else k
                    if (model_cfg.name, dataset_name, condition, int(seed), combo_k) in completed_combos:
                        continue
                    batch_rows = []
                    prompts = []
                    for environment, test_df in [('id', test_id), ('ood', test_ood)]:
                        for query_id, (_, query) in enumerate(test_df.iterrows()):
                            demo_ids = select_demos(condition, train_pool, query, k, seed, feature_cols, artifacts, environment, query_id)
                            demo_lines = build_demo_lines(train_pool, demo_ids, feature_cols, codebook)
                            query_line = build_query_line(query, feature_cols, codebook)
                            prompt = build_classification_prompt(
                                task_description, label_tokens, demo_lines, query_line,
                                label_meanings=(label_meaning_0, label_meaning_1),
                            )
                            prompts.append(prompt)
                            batch_rows.append({
                                'arm': 'real', 'dataset': dataset_name, 'environment': environment,
                                'model': model_cfg.name, 'method': condition, 'seed': int(seed),
                                'query_id': query_id, 'label': str(int(query['label'])),
                                'demo_ids': [int(i) for i in demo_ids], 'k': len(demo_ids),
                            })

                    predictions = runner.batch_predict(prompts, label_tokens)
                    for row, pred in zip(batch_rows, predictions):
                        row['prediction'] = pred.prediction
                        row['logprob_0'] = pred.logprob_0
                        row['logprob_1'] = pred.logprob_1

                    append_results(pd.DataFrame(batch_rows), RESULTS_PATH)
                    completed_combos.add((model_cfg.name, dataset_name, condition, int(seed), combo_k))
                    tqdm.write(f"{model_cfg.name} | {dataset_name} | k={k} | {condition} | seed={seed}: {len(batch_rows)} rows")

    runner.shutdown()

Models:   0%|          | 0/2 [00:00<?, ?it/s]

Models:   0%|          | 0/2 [00:00<?, ?it/s, model=Llama-3.1-8B-Instruct]

INFO 09-12 09:16:17 [api_utils.py:286] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
WARNING 09-12 09:16:18 [arg_utils.py:1801] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-12 09:16:18 [model.py:684] Resolved architecture: LlamaForCausalLM
INFO 09-12 09:16:18 [model.py:2021] Using max model len 8192
INFO 09-12 09:16:18 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 09-12 09:16:18 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


INFO 09-12 09:16:20 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoi

...

Datasets: 100%|██████████| 4/4 [06:29<00:00, 153.48s/it, dataset=anes]

Qwen2.5-7B-Instruct | anes | k=16 | counter_spurious | seed=1024: 1000 rows


[rank0]:[W912 09:24:57.672906075 ProcessGroupNCCL.cpp:1624] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Models: 100%|██████████| 2/2 [08:48<00:00, 300.37s/it, model=Qwen2.5-7B-Instruct]

Models: 100%|██████████| 2/2 [08:48<00:00, 264.08s/it, model=Qwen2.5-7B-Instruct]

## Output

- `results/real_arm_baselines.parquet`
- Summary table: accuracy, macro-F1, shift gap per (dataset, model, condition), averaged over seeds with std.

In [4]:
from src.evaluation.accuracy import summarise
from src.utils.results_schema import load_results

try:
    results = load_results(RESULTS_PATH)
except FileNotFoundError:
    results = None

if results is None or results.empty:
    print("No results yet — results/real_arm_baselines.parquet doesn't exist "
          "(the inference cell above was skipped since vLLM isn't available here).")
    summary = pd.DataFrame(columns=[
        'dataset', 'model', 'method', 'environment', 'k', 'accuracy_mean', 'accuracy_std',
        'macro_f1_mean', 'macro_f1_std', 'invalid_rate_mean', 'n', 'shift_gap',
    ])
else:
    # 'k' is included in every group_cols/groupby below -- without it, the
    # k_primary and k_sensitivity sweep rows would get silently averaged
    # together into one number per (dataset, model, method, environment),
    # which would blend two different experiments instead of comparing them.
    # Per-seed accuracy/macro-F1 first, then mean +/- std *across seeds* (not a
    # single pooled accuracy over all seeds' rows).
    per_seed = summarise(results, group_cols=['dataset', 'model', 'method', 'environment', 'k', 'seed'])
    summary = per_seed.groupby(['dataset', 'model', 'method', 'environment', 'k']).agg(
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        invalid_rate_mean=('invalid_rate', 'mean'),
        n=('n', 'sum'),
    ).reset_index()

    shift_gap = summary.pivot_table(index=['dataset', 'model', 'method', 'k'], columns='environment', values='accuracy_mean')
    shift_gap = (shift_gap['id'] - shift_gap['ood']).rename('shift_gap').reset_index()
    summary = summary.merge(shift_gap, on=['dataset', 'model', 'method', 'k'], how='left')

# Saved separately from the schema'd per-prediction parquet — Notebook 03 reads
# this to pick the "best-performing protocol from Notebook 02" per (dataset, model),
# filtered to k == config.k_primary (see Notebook 03's best_protocol_for).
summary.to_parquet(resolve_path('results/real_arm_baselines_summary.parquet'), index=False)

# k-sensitivity view: does the k=8 ranking of conditions hold at k=16? Printed
# separately rather than folded into `summary` above so the headline
# k_primary table (what every other notebook consumes) stays exactly the
# shape it was before this sweep was added.
if not summary.empty and len(summary['k'].unique()) > 1:
    k_sensitivity = summary.pivot_table(index=['dataset', 'model', 'method'], columns='k', values='accuracy_mean')
    print("k-sensitivity (OOD+ID accuracy_mean, k_primary vs k_sensitivity):")
    display(k_sensitivity)

summary

k-sensitivity (OOD+ID accuracy_mean, k_primary vs k_sensitivity):


k                                                         0       8       16
dataset        model                 method                                 
acsincome      Llama-3.1-8B-Instruct counter_spurious    NaN  0.5968  0.6256
                                     feature_range       NaN  0.5688  0.6156
                                     label_diversity     NaN  0.6848  0.6684
                                     random              NaN  0.6304  0.6576
                                     rule_diversity      NaN  0.6456  0.6908
                                     similarity          NaN  0.6090  0.6560
                                     zero_shot         0.624     NaN     NaN
               Qwen2.5-7B-Instruct   counter_spurious    NaN  0.7044  0.7214
                                     feature_range       NaN  0.7192  0.6932
                                     label_diversity     NaN  0.6924  0.7396
                                     random              NaN  0.7172  0.7040
                                     rule_diversity      NaN  0.7202  0.7240
                                     similarity          NaN  0.6980  0.6930
                                     zero_shot         0.689     NaN     NaN
acspubcov      Llama-3.1-8B-Instruct counter_spurious    NaN  0.5102  0.4658
                                     feature_range       NaN  0.4970  0.4380
                                     label_diversity     NaN  0.5086  0.5500
                                     random              NaN  0.4562  0.4854
                                     rule_diversity      NaN  0.5012  0.4954
                                     similarity          NaN  0.5070  0.4700
                                     zero_shot         0.598     NaN     NaN
               Qwen2.5-7B-Instruct   counter_spurious    NaN  0.4520  0.4658
                                     feature_range       NaN  0.4794  0.5146
                                     label_diversity     NaN  0.4968  0.5340
                                     random              NaN  0.4576  0.5008
                                     rule_diversity      NaN  0.4616  0.5024
                                     similarity          NaN  0.5210  0.5500
                                     zero_shot         0.429     NaN     NaN
anes           Llama-3.1-8B-Instruct counter_spurious    NaN  0.6714  0.6638
                                     feature_range       NaN  0.7090  0.7310
                                     label_diversity     NaN  0.6312  0.6600
                                     random              NaN  0.7204  0.7442
                                     rule_diversity      NaN  0.7038  0.7062
                                     similarity          NaN  0.6130  0.6870
                                     zero_shot         0.320     NaN     NaN
               Qwen2.5-7B-Instruct   counter_spurious    NaN  0.6782  0.6126
                                     feature_range       NaN  0.6818  0.6926
                                     label_diversity     NaN  0.6804  0.6410
                                     random              NaN  0.6682  0.6094
                                     rule_diversity      NaN  0.6750  0.6384
                                     similarity          NaN  0.6120  0.6280
                                     zero_shot         0.529     NaN     NaN
brfss_diabetes Llama-3.1-8B-Instruct counter_spurious    NaN  0.3432  0.4612
                                     feature_range       NaN  0.6182  0.6370
                                     label_diversity     NaN  0.6936  0.6636
                                     random              NaN  0.5524  0.5256
                                     rule_diversity      NaN  0.4080  0.5300
                                     similarity          NaN  0.4660  0.5240
                                     zero_shot         0.427     NaN     NaN
               Qwen2.5-7B-Instruct   counter_spurious    NaN  0.7154  0.7

,dataset,model,method,environment,k,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,invalid_rate_mean,n,shift_gap
0,acsincome,Llama-3.1-8B-Instruct,counter_spurious,id,8,0.5848,0.159013,0.563731,0.188627,0.0,2500,-0.0240
1,acsincome,Llama-3.1-8B-Instruct,counter_spurious,id,16,0.6132,0.101485,0.601681,0.100636,0.0,2500,-0.0248
2,acsincome,Llama-3.1-8B-Instruct,counter_spurious,ood,8,0.6088,0.131929,0.583539,0.177532,0.0,2500,-0.0240
3,acsincome,Llama-3.1-8B-Instruct,counter_spurious,ood,16,0.6380,0.086394,0.625771,0.098578,0.0,2500,-0.0248
4,acsincome,Llama-3.1-8B-Instruct,feature_range,id,8,0.5412,0.126662,0.526379,0.139671,0.0,2500,-0.0552
...,...,...,...,...,...,...,...,...,...,...,...,...
203,brfss_diabetes,Qwen2.5-7B-Instruct,similarity,id,16,0.6760,0.000000,0.521684,0.000000,0.0,2500,-0.0560
204,brfss_diabetes,Qwen2.5-7B-Instruct,similarity,ood,8,0.7380,0.000000,0.639870,0.000000,0.0,2500,-0.0720
205,brfss_diabetes,Qwen2.5-7B-Instruct,similarity,ood,16,0.7320,0.000000,0.628405,0.000000,0.0,2500,-0.0560
206,brfss_diabetes,Qwen2.5-7B-Instruct,zero_shot,id,0,0.7840,0.000000,0.561916,0.000000,0.0,2500,0.0040
